# H1. A public fill-probability task (analog of the HSBC/IBM RFQ data)

The HSBC/IBM paper (Ciceri et al., arXiv:2509.17715) predicts whether a dealer's quote is filled, from 216 features
of the request and the market. Their data are private. This notebook builds a public analog from Binance USD-M
futures (data.binance.vision):

| | HSBC/IBM | Here |
|---|---|---|
| Event | request for quote on a European corporate bond | passive limit order on a crypto perpetual future |
| Label | quote accepted (1) or not (0) | touched within 60 s (1) or not (0) |
| Fill rate | about 37% | 37.6% on day 1 (the quote distance is set to hit 37%) |
| Features | 216: request, ticker and sector level, short and long lags | 216: own symbol, market (BTC, ETH), cross-section, depth, positioning, event |
| Sampling | 16,000 events at random times, 69 days | a pool of 4,000 events: 500 at random times per day, 8 days (Sep 1 to 8, 2026), 8 symbols |

**Pool versus hardware events.** This notebook builds the full pool of 4,000 events. The hardware study does not use
all of them: H2 draws **1,000 events (125 per market day)** from this pool, and those same 1,000 events go into every
hardware arm (R, C and N, each at 1,024 shots). The pool is used here only for the classical reference baseline.

Labels use the standard "touch" rule of fill-probability studies: a resting buy at price p is filled if a trade
prints at or below p within the horizon (queue position ignored).

Run time: seconds with the delivered tables. Rebuilding from scratch (RECOMPUTE = True) downloads about 420 MB and takes about 10 minutes.

In [ ]:
DAYS        = ["2026-08-31"] + [f"2026-09-0{d}" for d in range(1, 9)]    # first day = warm-up for 1-hour features
EVENTS_DAY  = 500
HORIZON_S   = 60
TARGET_FILL = 0.37          # HSBC's fill rate; sets the quote distance on day 1 only
SEED        = 2026
RECOMPUTE   = False
RUN_BACKTESTS = False       # True reruns the classical and time-only backtests (about 30 min); else read saved

In [ ]:
import os, sys, json, time, math, warnings
from pathlib import Path
# Find the study folder (the one holding the hsbcrep package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd()]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "hsbcrep" / "__init__.py").exists():
                return p.resolve()
    raise RuntimeError(f"Cannot find the repository folder (the one holding hsbcrep/) from {Path.cwd()}. Set ROOT by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repository folder:", ROOT)
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
RAW = ROOT / "data" / "raw"
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
try:
    import xgboost
except ImportError:
    %pip install -q xgboost
from hsbcrep import cryptofill as CF
from hsbcrep import backtest as BT

## H1.1 Download

In [ ]:
# Only needed to rebuild the event and feature tables; the delivered tables in outputs/ are used otherwise.
if RECOMPUTE or not (OUT / "events.csv").exists():
    missing = CF.download(RAW, DAYS)
    print(f"{len(list(RAW.glob('*.zip')))} files in {RAW}; missing: {missing or 'none'}")
else:
    print("using the saved tables in outputs/ (set RECOMPUTE = True to download and rebuild)")

## H1.2 Events, labels and features

In [ ]:
f_ev, f_F = OUT / "events.csv", OUT / "features.csv"
if f_ev.exists() and f_F.exists() and not RECOMPUTE:
    ev, F = pd.read_csv(f_ev), pd.read_csv(f_F)
    rule = json.loads((OUT / "label_rule.json").read_text())
else:
    t0 = time.time()
    trades = {s: CF.load_trades(RAW, s, DAYS) for s in CF.SYMBOLS}
    depth = {s: CF.load_depth(RAW, s, DAYS) for s in CF.SYMBOLS}
    metrics = {s: CF.load_metrics(RAW, s, DAYS) for s in CF.SYMBOLS}
    ev = CF.sample_events(DAYS[1:], EVENTS_DAY, seed=SEED)
    day1 = ev[ev.day == DAYS[1]].reset_index(drop=True)
    delta, rate = CF.calibrate_delta(day1, trades, HORIZON_S, TARGET_FILL)
    ev["y"] = CF.label_events(ev, trades, delta, HORIZON_S)
    F = CF.features(ev, trades, depth, metrics)
    rule = {"delta_bps": delta, "horizon_s": HORIZON_S, "day1_rate": rate}
    ev.to_csv(f_ev, index=False); F.to_csv(f_F, index=False); (OUT / "label_rule.json").write_text(json.dumps(rule))
    print(f"built in {time.time() - t0:.0f} s")
print(f"{len(ev)} events, {F.shape[1]} features; quote distance {rule['delta_bps']} bps, horizon {rule['horizon_s']} s")
print("fill rate by day:", ev.groupby("day").y.mean().round(3).to_dict())
print("fill rate by symbol:", ev.groupby("symbol").y.mean().round(3).to_dict())

## H1.3 Baselines under HSBC's rolling backtest

On the full pool of 4,000 events (not the 1,000 hardware events; their baselines are in H3): training window 500
events, test blocks of 100, blinding window 0 or 1 day; logistic regression and gradient boosting. Two inputs: the
216 classical features, and the event time alone. The time-only model measures how much
a pure clock can earn in a forward rolling backtest. (HSBC report classical AUC 0.63 at 0 days.)

In [ ]:
f_bt = OUT / "baselines_pool.csv"
if f_bt.exists() and not RUN_BACKTESTS:
    bt = pd.read_csv(f_bt)
else:
    o = np.argsort(ev.t.values, kind="stable"); evs, Fs = ev.iloc[o].reset_index(drop=True), F.iloc[o].reset_index(drop=True)
    fams = BT.families(); fast = {k: fams[k] for k in ("LR", "XGB")}
    rows = []
    for name, X in (("classical 216 features", Fs.values), ("time only", ((evs.t - evs.t.min()) / 8.64e7).values[:, None])):
        for blind in (0, 1):
            p = BT.rolling(X, evs.y.values, evs.t.values, evs.day.values, n_train=500, block=100, blind_days=blind, fams=fast)
            rows.append({"input": name, "blind_days": blind, **BT.summarize(p)})
    bt = pd.DataFrame(rows); bt.to_csv(f_bt, index=False)
bt.round(3)